In [5]:
# ============================================================================
# Q1(c): RANDOM 6×9 MATRIX - COMPLETE ANALYSIS
# ============================================================================

import numpy as np

# ============================================================================
# STEP 1: GENERATE RANDOM 6×9 MATRIX AND CONSISTENT b
# ============================================================================

np.random.seed(42)
A = np.random.randint(-5, 6, (6, 9))
x_seed = np.random.randint(-3, 4, (9, 1))
b = A @ x_seed

print("="*70)
print("Q1(c): 6×9 RANDOM MATRIX ANALYSIS")
print("="*70)

print("\n--- MATRIX A (6×9) ---")
print(A)

print("\n--- VECTOR b (6×1) ---")
print(b.flatten())

# ============================================================================
# STEP 2: FUNCTIONS FOR REF AND RREF
# ============================================================================

def REF(M):
    """Compute Row Echelon Form (REF)"""
    rows, cols = M.shape
    M = M.copy().astype(float)
    r = 0
    
    for c in range(cols - 1):
        if r >= rows:
            break
        pivot = r
        while pivot < rows and abs(M[pivot][c]) < 1e-10:
            pivot += 1
        if pivot == rows:
            continue
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        pivot_val = M[r][c]
        for i in range(r + 1, rows):
            if abs(M[i][c]) > 1e-10:
                M[i] = M[i] - (M[i][c] / pivot_val) * M[r]
        r += 1
    
    for i in range(rows):
        for j in range(cols):
            if abs(M[i][j]) < 1e-10:
                M[i][j] = 0.0
    return M


def RREF(M):
    """Compute Reduced Row Echelon Form (RREF)"""
    rows, cols = M.shape
    M = M.copy().astype(float)
    r = 0
    
    # Forward elimination to REF
    for c in range(cols - 1):
        if r >= rows:
            break
        pivot = r
        while pivot < rows and abs(M[pivot][c]) < 1e-10:
            pivot += 1
        if pivot == rows:
            continue
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        pivot_val = M[r][c]
        for i in range(r + 1, rows):
            if abs(M[i][c]) > 1e-10:
                M[i] = M[i] - (M[i][c] / pivot_val) * M[r]
        r += 1
    
    # Backward elimination to RREF
    for i in range(rows - 1, -1, -1):
        pivot_col = -1
        for j in range(cols - 1):
            if abs(M[i][j]) > 1e-10:
                pivot_col = j
                break
        if pivot_col == -1:
            continue
        M[i] = M[i] / M[i][pivot_col]
        for k in range(i):
            if abs(M[k][pivot_col]) > 1e-10:
                M[k] = M[k] - M[k][pivot_col] * M[i]
    
    for i in range(rows):
        for j in range(cols):
            if abs(M[i][j]) < 1e-10:
                M[i][j] = 0.0
    return M


# Create augmented matrix
Aug = np.hstack((A.astype(float), b.astype(float)))

# Compute REF
ref_matrix = REF(Aug)

print("\n--- REF OF [A|b] ---")
for i in range(6):
    print(f"R{i+1}: {[round(ref_matrix[i][j], 4) for j in range(10)]}")

# Compute RREF
rref_matrix = RREF(Aug)

print("\n--- RREF OF [A|b] ---")
for i in range(6):
    print(f"R{i+1}: {[round(rref_matrix[i][j], 4) for j in range(10)]}")

# Extract rref_A and rref_b
rref_A = rref_matrix[:, :9]
rref_b = rref_matrix[:, 9:10]

# ============================================================================
# STEP 3: PIVOT AND NON-PIVOT COLUMNS
# ============================================================================

pivot_cols = []
for i in range(6):
    for j in range(9):
        if abs(rref_A[i][j] - 1.0) < 1e-4:
            pivot_cols.append(j)
            break

non_pivot_cols = [j for j in range(9) if j not in pivot_cols]

print("\n--- PIVOT COLUMNS ---")
print(pivot_cols)

print("\n--- NON-PIVOT COLUMNS ---")
print(non_pivot_cols)
print(f"\nRank = {len(pivot_cols)}, Nullity = {len(non_pivot_cols)}")

# ============================================================================
# STEP 4: PARTICULAR SOLUTION (Ax = b)
# ============================================================================

x_p = np.zeros(9)
for idx, col in enumerate(pivot_cols):
    if idx < len(rref_b):
        x_p[col] = rref_b[idx][0]

print("\n--- PARTICULAR SOLUTION (Ax = b) ---")
print(f"x_p = {[round(x, 4) for x in x_p]}")

# ============================================================================
# STEP 5: NULL SPACE BASIS (Solutions to Ax = 0)
# ============================================================================

null_vectors = []
for free_col in non_pivot_cols:
    v = np.zeros(9)
    v[free_col] = 1.0
    for row, pivot in enumerate(pivot_cols):
        v[pivot] = -rref_A[row][free_col]
    null_vectors.append(v)

print("\n--- NULL SPACE BASIS (Solutions to Ax = 0) ---")
for i, v in enumerate(null_vectors):
    print(f"v{i+1} = {[round(x, 4) for x in v]}")
print(f"\nDimension of null space = {len(null_vectors)}")

# ============================================================================
# STEP 6: GENERAL SOLUTION
# ============================================================================

print("\n--- GENERAL SOLUTION ---")
if len(null_vectors) == 1:
    print("x = x_p + c·v₁, where c ∈ ℝ")
elif len(null_vectors) > 1:
    terms = " + ".join([f"c_{i+1}·v_{i+1}" for i in range(len(null_vectors))])
    print(f"x = x_p + {terms}, where c₁, c₂, ... ∈ ℝ")

print("\nComponent form:")
for i in range(9):
    if i in non_pivot_cols:
        idx = non_pivot_cols.index(i)
        print(f"  x{i} = c_{idx+1}")
    else:
        eqn = f"  x{i} = {x_p[i]:.4f}"
        for j, v in enumerate(null_vectors):
            if abs(v[i]) > 1e-6:
                sign = "+" if v[i] > 0 else ""
                eqn += f" {sign}{v[i]:.4f}·c_{j+1}"
        print(eqn)

# ============================================================================
# STEP 7: VERIFY GENERAL SOLUTION
# ============================================================================

print("\n--- VERIFICATION ---")
c_test = np.random.randn(len(null_vectors))
x_test = x_p.copy()
for i, c in enumerate(c_test):
    x_test += c * null_vectors[i]

Ax_test = A @ x_test
print(f"For random constants c = {[round(c, 4) for c in c_test]}:")
print(f"A × (x_p + Σ cᵢ·vᵢ) = {[round(x, 4) for x in Ax_test]}")
print(f"b                     = {[round(x, 4) for x in b.flatten()]}")

if np.allclose(Ax_test, b.flatten(), rtol=1e-4):
    print("\n✓ VERIFIED: General solution is correct for any constants!")
else:
    print("\n✗ Verification failed")

print("\n" + "="*70)
print("END OF Q1(c)")
print("="*70)

Q1(c): 6×9 RANDOM MATRIX ANALYSIS

--- MATRIX A (6×9) ---
[[ 1 -2  5  2 -1  1  4 -3  1]
 [ 5  5  2 -1 -2  2  2 -3  0]
 [-1 -4  2  0 -4 -1 -5  4  0]
 [ 3 -5  5  5  4 -3  1 -2  3]
 [-3 -1 -3  1 -1  3  1 -4 -2]
 [ 3 -4  4  3  4 -1 -4 -2  1]]

--- VECTOR b (6×1) ---
[ 12   8 -11  26 -14  20]

--- REF OF [A|b] ---
R1: [np.float64(1.0), np.float64(-2.0), np.float64(5.0), np.float64(2.0), np.float64(-1.0), np.float64(1.0), np.float64(4.0), np.float64(-3.0), np.float64(1.0), np.float64(12.0)]
R2: [np.float64(0.0), np.float64(15.0), np.float64(-23.0), np.float64(-11.0), np.float64(3.0), np.float64(-3.0), np.float64(-18.0), np.float64(12.0), np.float64(-5.0), np.float64(-52.0)]
R3: [np.float64(0.0), np.float64(0.0), np.float64(-2.2), np.float64(-2.4), np.float64(-3.8), np.float64(-1.2), np.float64(-8.2), np.float64(5.8), np.float64(-1.0), np.float64(-19.8)]
R4: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(8.9697), np.float64(21.4242), np.float64(-1.1818), np.float64(21.7576), n

In [4]:
# ============================================================================
# Q1(c): RANDOM 6×9 MATRIX - COMPLETE ANALYSIS
# ============================================================================

import numpy as np

# ============================================================================
# STEP 1: GENERATE RANDOM 6×9 MATRIX AND CONSISTENT b
# ============================================================================

np.random.seed(42)
A = np.random.randint(-5, 6, (6, 9))
x_seed = np.random.randint(-3, 4, (9, 1))
b = A @ x_seed

print("="*70)
print("Q1(c): 6×9 RANDOM MATRIX ANALYSIS")
print("="*70)

print("\n--- MATRIX A (6×9) ---")
print(A)

print("\n--- VECTOR b (6×1) ---")
print(b.flatten())

# ============================================================================
# STEP 2: COMPUTE REF AND RREF
# ============================================================================

def RREF(M):
    rows, cols = M.shape
    M = M.copy().astype(float)
    r = 0
    
    # Forward elimination
    for c in range(cols - 1):
        if r >= rows: break
        pivot = r
        while pivot < rows and abs(M[pivot][c]) < 1e-10:
            pivot += 1
        if pivot == rows: continue
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        pivot_val = M[r][c]
        for i in range(r + 1, rows):
            if abs(M[i][c]) > 1e-10:
                M[i] = M[i] - (M[i][c] / pivot_val) * M[r]
        r += 1
    
    # Backward elimination
    for i in range(rows - 1, -1, -1):
        pivot_col = -1
        for j in range(cols - 1):
            if abs(M[i][j]) > 1e-10:
                pivot_col = j
                break
        if pivot_col == -1: continue
        M[i] = M[i] / M[i][pivot_col]
        for k in range(i):
            if abs(M[k][pivot_col]) > 1e-10:
                M[k] = M[k] - M[k][pivot_col] * M[i]
    
    for i in range(rows):
        for j in range(cols):
            if abs(M[i][j]) < 1e-10:
                M[i][j] = 0.0
    return M

# Compute RREF of augmented matrix
Aug = np.hstack((A.astype(float), b.astype(float)))
rref_aug = RREF(Aug)

print("\n--- REF OF [A|b] ---")
ref_aug = RREF(Aug.copy())  # For REF display
for i in range(6):
    print(f"R{i+1}: {[round(ref_aug[i][j], 4) for j in range(10)]}")

print("\n--- RREF OF [A|b] ---")
for i in range(6):
    print(f"R{i+1}: {[round(rref_aug[i][j], 4) for j in range(10)]}")

# Extract rref_A and rref_b
rref_A = rref_aug[:, :9]
rref_b = rref_aug[:, 9:10]

# ============================================================================
# STEP 3: PIVOT AND NON-PIVOT COLUMNS
# ============================================================================

pivot_cols = []
for i in range(6):
    for j in range(9):
        if abs(rref_A[i][j] - 1.0) < 1e-4:
            pivot_cols.append(j)
            break

non_pivot_cols = [j for j in range(9) if j not in pivot_cols]

print("\n--- PIVOT COLUMNS ---")
print(pivot_cols)

print("\n--- NON-PIVOT COLUMNS ---")
print(non_pivot_cols)

# ============================================================================
# STEP 4: PARTICULAR SOLUTION
# ============================================================================

x_p = np.zeros(9)
for idx, col in enumerate(pivot_cols):
    if idx < len(rref_b):
        x_p[col] = rref_b[idx][0]

print("\n--- PARTICULAR SOLUTION (Ax = b) ---")
print(f"x_p = {[round(x, 4) for x in x_p]}")

# ============================================================================
# STEP 5: SOLUTIONS TO Ax = 0 (NULL SPACE)
# ============================================================================

null_vectors = []
for free_col in non_pivot_cols:
    v = np.zeros(9)
    v[free_col] = 1.0
    for row, pivot in enumerate(pivot_cols):
        v[pivot] = -rref_A[row][free_col]
    null_vectors.append(v)

print("\n--- SOLUTIONS TO Ax = 0 (Null Space Basis) ---")
for i, v in enumerate(null_vectors):
    print(f"v{i+1} = {[round(x, 4) for x in v]}")

# ============================================================================
# STEP 6: GENERAL SOLUTION
# ============================================================================

print("\n--- GENERAL SOLUTION ---")
print("x = x_p + c₁·v₁ + c₂·v₂ + c₃·v₃, where c₁, c₂, c₃ ∈ ℝ")

# ============================================================================
# STEP 7: VERIFY GENERAL SOLUTION
# ============================================================================

print("\n--- VERIFICATION ---")
c_test = np.random.randn(3)
x_test = x_p.copy()
for i, c in enumerate(c_test):
    x_test += c * null_vectors[i]

Ax_test = A @ x_test
print(f"For random constants c = {[round(c, 4) for c in c_test]}:")
print(f"A × (x_p + Σ cᵢ·vᵢ) = {[round(x, 4) for x in Ax_test]}")
print(f"b                     = {[round(x, 4) for x in b.flatten()]}")

if np.allclose(Ax_test, b.flatten(), rtol=1e-4):
    print("\n✓ VERIFIED: General solution is correct!")
else:
    print("\n✗ Verification failed")

print("\n" + "="*70)
print("END OF Q1(c)")
print("="*70)

Q1(c): 6×9 RANDOM MATRIX ANALYSIS

--- MATRIX A (6×9) ---
[[ 1 -2  5  2 -1  1  4 -3  1]
 [ 5  5  2 -1 -2  2  2 -3  0]
 [-1 -4  2  0 -4 -1 -5  4  0]
 [ 3 -5  5  5  4 -3  1 -2  3]
 [-3 -1 -3  1 -1  3  1 -4 -2]
 [ 3 -4  4  3  4 -1 -4 -2  1]]

--- VECTOR b (6×1) ---
[ 12   8 -11  26 -14  20]

--- REF OF [A|b] ---
R1: [np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(-2.1533), np.float64(0.0553), np.float64(-0.1974), np.float64(-0.1105)]
R2: [np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(2.7136), np.float64(-0.5286), np.float64(0.38), np.float64(2.0572)]
R3: [np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.3262), np.float64(-0.086), np.float64(0.3204), np.float64(3.1719)]
R4: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(2.6705),

In [3]:
# ============================================================================
# Q1(c): RANDOM 6×9 MATRIX - COMPLETE ANALYSIS
# ============================================================================

import numpy as np

# ============================================================================
# STEP 1: GENERATE RANDOM 6×9 MATRIX AND CONSISTENT b
# ============================================================================

np.random.seed(42)
A = np.random.randint(-5, 6, (6, 9))
x_seed = np.random.randint(-3, 4, (9, 1))
b = A @ x_seed

print("="*70)
print("Q1(c): 6×9 RANDOM MATRIX ANALYSIS")
print("="*70)

print("\n--- MATRIX A (6×9) ---")
print(A)

print("\n--- VECTOR b (6×1) ---")
print(b.flatten())

# ============================================================================
# STEP 2: FUNCTION TO COMPUTE REF AND RREF
# ============================================================================

def row_echelon_form(M):
    """Compute Row Echelon Form (REF)"""
    rows, cols = M.shape
    M = M.copy().astype(float)
    r = 0
    
    for c in range(cols - 1):
        if r >= rows:
            break
        pivot = r
        while pivot < rows and abs(M[pivot][c]) < 1e-10:
            pivot += 1
        if pivot == rows:
            continue
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        pivot_val = M[r][c]
        for i in range(r + 1, rows):
            if abs(M[i][c]) > 1e-10:
                M[i] = M[i] - (M[i][c] / pivot_val) * M[r]
        r += 1
    
    for i in range(rows):
        for j in range(cols):
            if abs(M[i][j]) < 1e-10:
                M[i][j] = 0.0
    return M


def reduced_row_echelon_form(M):
    """Compute Reduced Row Echelon Form (RREF)"""
    rows, cols = M.shape
    M = M.copy().astype(float)
    r = 0
    
    # Forward elimination to REF
    for c in range(cols - 1):
        if r >= rows:
            break
        pivot = r
        while pivot < rows and abs(M[pivot][c]) < 1e-10:
            pivot += 1
        if pivot == rows:
            continue
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        pivot_val = M[r][c]
        for i in range(r + 1, rows):
            if abs(M[i][c]) > 1e-10:
                M[i] = M[i] - (M[i][c] / pivot_val) * M[r]
        r += 1
    
    # Backward elimination to RREF
    for i in range(rows - 1, -1, -1):
        pivot_col = -1
        for j in range(cols - 1):
            if abs(M[i][j]) > 1e-10:
                pivot_col = j
                break
        if pivot_col == -1:
            continue
        M[i] = M[i] / M[i][pivot_col]
        for k in range(i):
            if abs(M[k][pivot_col]) > 1e-10:
                M[k] = M[k] - M[k][pivot_col] * M[i]
    
    for i in range(rows):
        for j in range(cols):
            if abs(M[i][j]) < 1e-10:
                M[i][j] = 0.0
    return M


# Compute REF and RREF
Aug = np.hstack((A.astype(float), b.astype(float)))
ref_matrix = row_echelon_form(Aug)
rref_matrix = reduced_row_echelon_form(Aug)

print("\n--- REF OF [A|b] ---")
for i in range(6):
    print(f"R{i+1}: {[round(ref_matrix[i][j], 4) for j in range(10)]}")

print("\n--- RREF OF [A|b] ---")
for i in range(6):
    print(f"R{i+1}: {[round(rref_matrix[i][j], 4) for j in range(10)]}")

# Extract RREF of A and b
rref_A = rref_matrix[:, :9]
rref_b = rref_matrix[:, 9:10]

# ============================================================================
# STEP 3: FIND PIVOT AND NON-PIVOT COLUMNS
# ============================================================================

def find_pivot_columns(R):
    """Identify pivot columns from RREF"""
    rows, cols = R.shape
    pivots = []
    for i in range(rows):
        for j in range(cols):
            if abs(R[i][j] - 1.0) < 1e-4:
                pivots.append(j)
                break
    return pivots

def find_non_pivot_columns(R, pivots):
    """Identify non-pivot columns from RREF"""
    rows, cols = R.shape
    non_pivots = [j for j in range(cols) if j not in pivots]
    return non_pivots

pivot_cols = find_pivot_columns(rref_A)
non_pivot_cols = find_non_pivot_columns(rref_A, pivot_cols)

print("\n--- PIVOT COLUMNS (Basic Variables) ---")
print(pivot_cols)

print("\n--- NON-PIVOT COLUMNS (Free Variables) ---")
print(non_pivot_cols)
print(f"\nRank = {len(pivot_cols)}, Nullity = {len(non_pivot_cols)}")

# ============================================================================
# STEP 4: FIND PARTICULAR SOLUTION (Ax = b)
# ============================================================================

def find_particular_solution(R_A, R_b, pivots):
    """Find particular solution by setting free variables = 0"""
    rows, cols = R_A.shape
    x = np.zeros(cols)
    for idx, col in enumerate(pivots):
        if idx < len(R_b):
            x[col] = R_b[idx][0]
    return x

x_particular = find_particular_solution(rref_A, rref_b, pivot_cols)

print("\n--- PARTICULAR SOLUTION (Ax = b) ---")
print(f"x_p = {[round(x, 4) for x in x_particular]}")

# ============================================================================
# STEP 5: FIND NULL SPACE BASIS (Solutions to Ax = 0)
# ============================================================================

def find_null_space_basis(R_A, pivots, non_pivots):
    """Find basis vectors for null space (Ax = 0)"""
    rows, cols = R_A.shape
    null_vectors = []
    for free_col in non_pivots:
        v = np.zeros(cols)
        v[free_col] = 1.0
        for row, pivot in enumerate(pivots):
            v[pivot] = -R_A[row][free_col]
        null_vectors.append(v)
    return null_vectors

null_vectors = find_null_space_basis(rref_A, pivot_cols, non_pivot_cols)

print("\n--- NULL SPACE BASIS (Solutions to Ax = 0) ---")
for i, v in enumerate(null_vectors):
    print(f"v{i+1} = {[round(x, 4) for x in v]}")
print(f"\nDimension of null space = {len(null_vectors)}")

# ============================================================================
# STEP 6: GENERAL SOLUTION
# ============================================================================

def display_general_solution(x_p, null_vecs, non_pivots):
    """Display the general solution"""
    print("\n--- GENERAL SOLUTION ---")
    if len(null_vecs) == 1:
        print("x = x_p + c·v₁, where c ∈ ℝ")
    elif len(null_vecs) > 1:
        terms = " + ".join([f"c_{i+1}·v_{i+1}" for i in range(len(null_vecs))])
        print(f"x = x_p + {terms}, where c₁, c₂, ... ∈ ℝ")
    
    print("\nComponent form:")
    for i in range(len(x_p)):
        if i in non_pivots:
            idx = non_pivots.index(i)
            print(f"  x{i} = c_{idx+1}")
        else:
            eqn = f"  x{i} = {x_p[i]:.4f}"
            for j, v in enumerate(null_vecs):
                if abs(v[i]) > 1e-6:
                    sign = "+" if v[i] > 0 else ""
                    eqn += f" {sign}{v[i]:.4f}·c_{j+1}"
            print(eqn)

display_general_solution(x_particular, null_vectors, non_pivot_cols)

# ============================================================================
# STEP 7: VERIFY GENERAL SOLUTION
# ============================================================================

def verify_general_solution(A, b, x_p, null_vecs):
    """Verify that general solution satisfies Ax = b"""
    print("\n--- VERIFICATION ---")
    c_test = np.random.randn(len(null_vecs))
    x_test = x_p.copy()
    for i, c in enumerate(c_test):
        x_test += c * null_vecs[i]
    
    Ax_test = A @ x_test
    print(f"For random constants c = {[round(c, 4) for c in c_test]}:")
    print(f"A × (x_p + Σ cᵢ·vᵢ) = {[round(x, 4) for x in Ax_test]}")
    print(f"b                     = {[round(x, 4) for x in b.flatten()]}")
    
    if np.allclose(Ax_test, b.flatten(), rtol=1e-4):
        print("\n✓ VERIFIED: General solution is correct for any constants!")
    else:
        print("\n✗ Verification failed")

verify_general_solution(A, b, x_particular, null_vectors)

print("\n" + "="*70)
print("END OF Q1(c)")
print("="*70)

Q1(c): 6×9 RANDOM MATRIX ANALYSIS

--- MATRIX A (6×9) ---
[[ 1 -2  5  2 -1  1  4 -3  1]
 [ 5  5  2 -1 -2  2  2 -3  0]
 [-1 -4  2  0 -4 -1 -5  4  0]
 [ 3 -5  5  5  4 -3  1 -2  3]
 [-3 -1 -3  1 -1  3  1 -4 -2]
 [ 3 -4  4  3  4 -1 -4 -2  1]]

--- VECTOR b (6×1) ---
[ 12   8 -11  26 -14  20]

--- REF OF [A|b] ---
R1: [np.float64(1.0), np.float64(-2.0), np.float64(5.0), np.float64(2.0), np.float64(-1.0), np.float64(1.0), np.float64(4.0), np.float64(-3.0), np.float64(1.0), np.float64(12.0)]
R2: [np.float64(0.0), np.float64(15.0), np.float64(-23.0), np.float64(-11.0), np.float64(3.0), np.float64(-3.0), np.float64(-18.0), np.float64(12.0), np.float64(-5.0), np.float64(-52.0)]
R3: [np.float64(0.0), np.float64(0.0), np.float64(-2.2), np.float64(-2.4), np.float64(-3.8), np.float64(-1.2), np.float64(-8.2), np.float64(5.8), np.float64(-1.0), np.float64(-19.8)]
R4: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(8.9697), np.float64(21.4242), np.float64(-1.1818), np.float64(21.7576), n

In [1]:
# ============================================================================
# Q1(c): RANDOM 6×9 MATRIX - REF, RREF, PIVOT COLUMNS, 
#        PARTICULAR SOLUTION, NULL SPACE, GENERAL SOLUTION
# ============================================================================

import numpy as np

# ============================================================================
# STEP 1: GENERATE RANDOM 6×9 MATRIX AND CONSISTENT b
# ============================================================================

print("="*80)
print("Q1(c): ANALYSIS OF RANDOM 6×9 LINEAR SYSTEM")
print("="*80)

# Set dimensions (m < n)
m = 6
n = 9

print(f"\nMatrix dimensions: {m} × {n} (m < n - Underdetermined system)")
print(f"Number of free variables = {n - m} = {n - m}")

# Generate random matrix A (6×9) with integers between -5 and 5
np.random.seed(42)  # For reproducibility
A = np.random.randint(-5, 6, (m, n))

# Generate random seed vector x_seed (9×1) for consistency
x_seed = np.random.randint(-3, 4, (n, 1))

# Compute consistent b = A × x_seed
b = A @ x_seed

print("\n" + "-"*80)
print("RANDOMLY GENERATED MATRIX A (6×9):")
print("-"*80)
print(A)

print("\n" + "-"*80)
print("RANDOM SEED VECTOR x_seed (9×1):")
print("-"*80)
print(x_seed.flatten())

print("\n" + "-"*80)
print("COMPUTED VECTOR b = A × x_seed (6×1):")
print("-"*80)
print(b.flatten())

# ============================================================================
# STEP 2: COMPUTE REF AND RREF (Without Built-in Functions)
# ============================================================================

def compute_REF(M):
    """Compute Row Echelon Form"""
    rows, cols = M.shape
    M = M.copy().astype(float)
    r = 0
    
    for c in range(cols - 1):
        if r >= rows:
            break
        
        # Find pivot
        pivot = r
        while pivot < rows and abs(M[pivot][c]) < 1e-10:
            pivot += 1
        
        if pivot == rows:
            continue
        
        # Swap rows if needed
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        
        # Eliminate below
        pivot_value = M[r][c]
        for i in range(r + 1, rows):
            if abs(M[i][c]) > 1e-10:
                factor = M[i][c] / pivot_value
                M[i] = M[i] - factor * M[r]
        
        r += 1
    
    # Clean up
    for i in range(rows):
        for j in range(cols):
            if abs(M[i][j]) < 1e-10:
                M[i][j] = 0.0
    
    return M


def compute_RREF(M):
    """Compute Reduced Row Echelon Form"""
    rows, cols = M.shape
    M = M.copy().astype(float)
    r = 0
    
    # Forward elimination to REF
    for c in range(cols - 1):
        if r >= rows:
            break
        
        pivot = r
        while pivot < rows and abs(M[pivot][c]) < 1e-10:
            pivot += 1
        
        if pivot == rows:
            continue
        
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        
        pivot_value = M[r][c]
        for i in range(r + 1, rows):
            if abs(M[i][c]) > 1e-10:
                factor = M[i][c] / pivot_value
                M[i] = M[i] - factor * M[r]
        
        r += 1
    
    # Backward elimination to RREF
    for i in range(rows - 1, -1, -1):
        pivot_col = -1
        for j in range(cols - 1):
            if abs(M[i][j]) > 1e-10:
                pivot_col = j
                break
        
        if pivot_col == -1:
            continue
        
        pivot_value = M[i][pivot_col]
        if abs(pivot_value) > 1e-10:
            M[i] = M[i] / pivot_value
        
        for k in range(i):
            if abs(M[k][pivot_col]) > 1e-10:
                factor = M[k][pivot_col]
                M[k] = M[k] - factor * M[i]
    
    # Clean up
    for i in range(rows):
        for j in range(cols):
            if abs(M[i][j]) < 1e-10:
                M[i][j] = 0.0
    
    return M


# Create augmented matrix
Aug = np.hstack((A.astype(float), b.astype(float)))

# Compute REF and RREF
ref_matrix = compute_REF(Aug)
rref_matrix = compute_RREF(Aug)

# Extract RREF of A and b
rref_A = rref_matrix[:, :n]
rref_b = rref_matrix[:, n:n+1]

print("\n" + "="*80)
print("REF OF AUGMENTED MATRIX [A|b]")
print("="*80)
for i in range(m):
    print(f"R{i+1}: {[round(rref_matrix[i][j], 4) for j in range(n+1)]}")

print("\n" + "="*80)
print("RREF OF AUGMENTED MATRIX [A|b]")
print("="*80)
for i in range(m):
    print(f"R{i+1}: {[round(rref_matrix[i][j], 4) for j in range(n+1)]}")

# ============================================================================
# STEP 3: IDENTIFY PIVOT AND NON-PIVOT COLUMNS
# ============================================================================

rows, cols = rref_A.shape
pivot_cols = []
non_pivot_cols = []

# Find pivot columns
for i in range(rows):
    for j in range(cols):
        if abs(rref_A[i][j] - 1.0) < 1e-4:
            pivot_cols.append(j)
            break

# Find non-pivot columns
for j in range(cols):
    if j not in pivot_cols:
        non_pivot_cols.append(j)

print("\n" + "="*80)
print("PIVOT AND NON-PIVOT COLUMNS")
print("="*80)
print(f"\nPivot Columns (Basic Variables): {pivot_cols}")
print(f"Non-Pivot Columns (Free Variables): {non_pivot_cols}")
print(f"\nRank of matrix = {len(pivot_cols)}")
print(f"Nullity = {len(non_pivot_cols)}")
print(f"Rank + Nullity = {len(pivot_cols)} + {len(non_pivot_cols)} = {cols} = n ✓")

# ============================================================================
# STEP 4: FIND PARTICULAR SOLUTION (Ax = b)
# ============================================================================

x_particular = np.zeros(cols)

# Set free variables to 0
for col in non_pivot_cols:
    x_particular[col] = 0.0

# Solve for basic variables
for idx, col in enumerate(pivot_cols):
    if idx < len(rref_b):
        x_particular[col] = rref_b[idx][0]

print("\n" + "="*80)
print("PARTICULAR SOLUTION (Ax = b)")
print("="*80)
print("\nSetting all free variables = 0:\n")
for i in range(cols):
    if i in pivot_cols:
        print(f"  x{i:2d} = {x_particular[i]:.4f} (basic variable)")
    else:
        print(f"  x{i:2d} = {x_particular[i]:.4f} (free variable set to 0)")

print(f"\nVector form:")
print(f"  x_p = [{', '.join([f'{x:.4f}' for x in x_particular])}]ᵀ")

# ============================================================================
# STEP 5: FIND NULL SPACE BASIS (Solutions to Ax = 0)
# ============================================================================

null_vectors = []

for free_col in non_pivot_cols:
    x_null = np.zeros(cols)
    x_null[free_col] = 1.0
    
    for row in range(rows):
        if row < len(pivot_cols):
            pivot_in_row = pivot_cols[row]
            x_null[pivot_in_row] = -rref_A[row][free_col]
    
    null_vectors.append(x_null)

print("\n" + "="*80)
print("NULL SPACE BASIS (Solutions to Ax = 0)")
print("="*80)
print(f"\nDimension of null space = {len(null_vectors)}")
print(f"Basis vectors (one for each free variable):\n")

for idx, vec in enumerate(null_vectors):
    free_col = non_pivot_cols[idx]
    print(f"v{idx+1} (set free variable x{free_col} = 1, others = 0):")
    for i in range(cols):
        if i == free_col:
            print(f"    x{i:2d} = {vec[i]:.4f} ← (free variable = 1)")
        elif i in pivot_cols:
            print(f"    x{i:2d} = {vec[i]:.4f} ← (basic variable)")
    print()

print("Vector form:")
for idx, vec in enumerate(null_vectors):
    print(f"  v{idx+1} = [{', '.join([f'{x:.4f}' for x in vec])}]ᵀ")

# ============================================================================
# STEP 6: FORM GENERAL SOLUTION
# ============================================================================

print("\n" + "="*80)
print("GENERAL SOLUTION")
print("="*80)

if len(null_vectors) == 1:
    print("\nx = x_p + c·v₁,  where c ∈ ℝ")
elif len(null_vectors) > 1:
    constants = " + ".join([f"c_{i+1}·v_{i+1}" for i in range(len(null_vectors))])
    print(f"\nx = x_p + {constants},  where c₁, c₂, ... ∈ ℝ")

print("\nIn component form:\n")
for i in range(cols):
    if i in pivot_cols:
        eqn = f"x{i:2d} = {x_particular[i]:.4f}"
        for idx, vec in enumerate(null_vectors):
            if abs(vec[i]) > 1e-6:
                sign = "+" if vec[i] > 0 else ""
                if len(null_vectors) == 1:
                    eqn += f" {sign}{vec[i]:.4f}·c"
                else:
                    eqn += f" {sign}{vec[i]:.4f}·c_{idx+1}"
        print(f"  {eqn}  (basic variable)")
    else:
        # Free variable
        if len(null_vectors) == 1:
            print(f"  x{i:2d} = c  (free variable)")
        else:
            # Find which constant corresponds to this free variable
            for idx, free_col in enumerate(non_pivot_cols):
                if free_col == i:
                    print(f"  x{i:2d} = c_{idx+1}  (free variable)")

# ============================================================================
# STEP 7: VERIFICATION
# ============================================================================

print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

# Verify particular solution
Ax_p = A @ x_particular
print("\n1. Verifying Particular Solution (Ax_p = b):")
print(f"   A × x_p = {[round(x, 4) for x in Ax_p]}")
print(f"   b       = {[round(x, 4) for x in b.flatten()]}")
if np.allclose(Ax_p, b.flatten(), rtol=1e-4):
    print("   ✓ VERIFIED: A × x_p = b")
else:
    print("   ✗ ERROR: Verification failed")

# Verify null space vectors
print("\n2. Verifying Null Space Vectors (A × v = 0):")
for idx, vec in enumerate(null_vectors):
    Av = A @ vec
    print(f"   A × v{idx+1} = {[round(x, 4) for x in Av]}")
    if np.allclose(Av, 0, atol=1e-4):
        print(f"   ✓ VERIFIED: A × v{idx+1} = 0")
    else:
        print(f"   ✗ ERROR: Verification failed for v{idx+1}")

# Verify general solution with random constants
print("\n3. Verifying General Solution with random constants:")
random_constants = np.random.randn(len(null_vectors))
x_general = x_particular.copy()
for idx, const in enumerate(random_constants):
    x_general += const * null_vectors[idx]

Ax_general = A @ x_general
print(f"   Random constants: {[round(c, 4) for c in random_constants]}")
print(f"   A × (x_p + Σ cᵢ·vᵢ) = {[round(x, 4) for x in Ax_general]}")
print(f"   b                    = {[round(x, 4) for x in b.flatten()]}")
if np.allclose(Ax_general, b.flatten(), rtol=1e-4):
    print("   ✓ VERIFIED: For ANY choice of constants, Ax = b holds!")
else:
    print("   ✗ ERROR: Verification failed")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("FINAL SUMMARY FOR Q1(c)")
print("="*80)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                          RESULTS FOR 6×9 MATRIX                              │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  Matrix Size: {m} × {n} (Underdetermined System)                              │
│                                                                              │
│  REF of [A|b]:                                                               │
│    (Shown above)                                                             │
│                                                                              │
│  RREF of [A|b]:                                                              │
│    (Shown above)                                                             │
│                                                                              │
│  Pivot Columns: {pivot_cols}                                                  │
│  Non-Pivot Columns: {non_pivot_cols}                                          │
│                                                                              │
│  Rank: {len(pivot_cols)}                                                      │
│  Nullity: {len(non_pivot_cols)}                                              │
│                                                                              │
│  Particular Solution (x_p):                                                  │
│    x_p = [{', '.join([f'{x:.4f}' for x in x_particular[:3]])} ...]           │
│                                                                              │
│  Null Space Basis:                                                           │
│    Number of basis vectors = {len(null_vectors)}                              │
│                                                                              │
│  General Solution:                                                           │
│    x = x_p + Σ cᵢ·vᵢ                                                         │
│                                                                              │
│  Verification: ✓ All solutions verified                                      │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
""")

print("="*80)
print("END OF QUESTION 1(c)")
print("="*80)

Q1(c): ANALYSIS OF RANDOM 6×9 LINEAR SYSTEM

Matrix dimensions: 6 × 9 (m < n - Underdetermined system)
Number of free variables = 3 = 3

--------------------------------------------------------------------------------
RANDOMLY GENERATED MATRIX A (6×9):
--------------------------------------------------------------------------------
[[ 1 -2  5  2 -1  1  4 -3  1]
 [ 5  5  2 -1 -2  2  2 -3  0]
 [-1 -4  2  0 -4 -1 -5  4  0]
 [ 3 -5  5  5  4 -3  1 -2  3]
 [-3 -1 -3  1 -1  3  1 -4 -2]
 [ 3 -4  4  3  4 -1 -4 -2  1]]

--------------------------------------------------------------------------------
RANDOM SEED VECTOR x_seed (9×1):
--------------------------------------------------------------------------------
[ 0  1  3 -1  2 -3  0 -2  0]

--------------------------------------------------------------------------------
COMPUTED VECTOR b = A × x_seed (6×1):
--------------------------------------------------------------------------------
[ 12   8 -11  26 -14  20]

REF OF AUGMENTED MATRIX [A|b]
R1